# Practice 03 — Complete Solution

---
## Step 1: Load the Raw Data

In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

with open('../data/raw_ab_data.json', 'r') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print("Initial DataFrame Shape:", df.shape)

---
## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
print(f"Missing values before: {df['Spend_USD'].isna().sum()}")
df = df.dropna(subset=['Spend_USD'])
print(f"Missing values after: {df['Spend_USD'].isna().sum()}")

In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=df['Spend_USD'], color='red')
plt.title("Extreme Whales Scewing the Data!")
plt.show()

In [ ]:
# 3. Winsorization (IQR Capping)
Q1 = df['Spend_USD'].quantile(0.25)
Q3 = df['Spend_USD'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

df.loc[df['Spend_USD'] > upper_bound, 'Spend_USD'] = upper_bound

plt.figure(figsize=(8,3))
sns.boxplot(x=df['Spend_USD'], color='green')
plt.title("Cleaned Data (Whales Capped)")
plt.show()

---
## Step 3: Statistical Causal Inference

In [ ]:
group_a = df[df['Group'] == 'A']['Converted']
group_b = df[df['Group'] == 'B']['Converted']

In [ ]:
cr_a = group_a.mean()
cr_b = group_b.mean()
print(f"CR A: {cr_a * 100:.2f}%")
print(f"CR B: {cr_b * 100:.2f}%")

In [ ]:
t_stat, p_val = stats.ttest_ind(group_b, group_a)

print(f"P Value: {p_val:.5f}")
if p_val < 0.05:
    print("Statistically Significant! Deploy Group B.")
else:
    print("Not Significant. Keep Group A.")